## Run ResNet-50 

**TASK:** Detect Insect Type \
**Dataset:** iNaturalist \
**Model:** Resnet-50 (pretrained)

In [1]:
import os
import warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # silence tensorflow logging
warnings.filterwarnings("ignore", category=UserWarning, module="PIL")

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from peft import LoraConfig, get_peft_model
from datasets import load_dataset
import seaborn as sns
import matplotlib.pyplot as plt

from utils.label_mappings import iNat_to_clean_map

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Data Loading

In [4]:
# custom mapping
class_dirs = ['Ant', 'Bee', 'Beetle', 'Butterfly', 'Grasshopper', 'Ladybug', 'Spider']
global_label_to_idx = {name: i for i, name in enumerate(class_dirs)}
num_classes = len(class_dirs)

In [5]:
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [6]:
ds = load_dataset("sxj1215/inaturalist", split='train')

def get_iNat_label(messages): 
    return messages[1]['content']

In [7]:
iNat36_label_df = pd.DataFrame({'messages': ds['messages']})
iNat36_label_df['species'] = iNat36_label_df['messages'].apply(get_iNat_label)
iNat36_label_df['clean_label'] = iNat36_label_df['species'].map(lambda x: iNat_to_clean_map.get(x, 'noise'))

In [8]:
# all valid insects for random/full experiments
all_valid_indices = iNat36_label_df[iNat36_label_df['clean_label'] != 'noise'].index.tolist()

In [9]:
# dataset classes

class InatPixelDataset(Dataset):
    def __init__(self, hf_dataset, indices, label_df, transform=None):
        self.hf_dataset, self.indices, self.label_df, self.transform = hf_dataset, indices, label_df, transform
    
    def __len__(self): 
        return len(self.indices)
    
    def __getitem__(self, idx):
        orig_idx = self.indices[idx]
        image = self.hf_dataset[orig_idx]['images'][0]
        label_str = self.label_df.iloc[orig_idx]['clean_label']
        if self.transform: 
            image = self.transform(image)
        return image, torch.tensor(global_label_to_idx[label_str], dtype=torch.long)

In [10]:
class KaggleTestDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths, self.transform = file_paths, transform
    
    def __len__(self): 
        return len(self.file_paths)
    
    def __getitem__(self, idx):
        path = self.file_paths[idx]
        label_name = os.path.basename(os.path.dirname(path))
        image = Image.open(path).convert('RGB')
        if self.transform: 
            image = self.transform(image)
        return image, torch.tensor(global_label_to_idx[label_name], dtype=torch.long)

In [11]:
# test set loading

test_base_path = 'data/clean_insect_images/'
test_files = []

for c in class_dirs:
    p = os.path.join(test_base_path, c)
    if os.path.exists(p):
        test_files.extend([os.path.join(p, f) for f in os.listdir(p) if f.lower().endswith(('.jpg', '.png'))])

test_loader = DataLoader(KaggleTestDataset(test_files, transform=preprocess), batch_size=32, shuffle=False)

### Model Functions

In [12]:
def get_partial_unfreeze_model():
    model = models.resnet50(weights="IMAGENET1K_V1")
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    
    for param in model.parameters(): 
        param.requires_grad = False
    
    for param in model.layer4.parameters(): 
        param.requires_grad = True
    
    for param in model.fc.parameters(): 
        param.requires_grad = True
        
    return model

In [13]:
def get_competitive_lora_model():
    model = models.resnet50(weights="IMAGENET1K_V1")
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    
    config = LoraConfig(r=32, lora_alpha=64, target_modules=["fc", "conv1", "downsample.0"], lora_dropout=0.1, bias="none")
    
    return get_peft_model(model, config)

In [14]:
# def run_experiment(name, model_fn):
#     print(f"\n{'='*20} {name} {'='*20}")
#     kf = KFold(n_splits=5, shuffle=True, random_state=1)
#     cv_accs = []
#     start_time = time.time()

#     for fold, (train_idx, val_idx) in enumerate(kf.split(train_val_dataset)):
#         t_loader = DataLoader(Subset(train_val_dataset, train_idx), batch_size=32, shuffle=True)
#         v_loader = DataLoader(Subset(train_val_dataset, val_idx), batch_size=32, shuffle=False)
        
#         model = model_fn().to(device)
#         optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
#         criterion = nn.CrossEntropyLoss()

#         for epoch in range(3):
#             model.train()
#             for imgs, labels in t_loader:
#                 imgs, labels = imgs.to(device), labels.to(device)
#                 optimizer.zero_grad(); loss = criterion(model(imgs), labels); loss.backward(); optimizer.step()
        
#         model.eval()
#         v_preds, v_targets = [], []
#         with torch.no_grad():
#             for imgs, labels in v_loader:
#                 out = model(imgs.to(device))
#                 v_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
#                 v_targets.extend(labels.numpy())
#         cv_accs.append(accuracy_score(v_targets, v_preds))
#         print(f"  [Fold {fold+1}] Val Acc: {cv_accs[-1]:.4f}")

#     # final test set eval
#     test_p, test_t = [], []
#     with torch.no_grad():
#         for imgs, labels in test_loader:
#             out = model(imgs.to(device))
#             test_p.extend(torch.argmax(out, dim=1).cpu().numpy())
#             test_t.extend(labels.numpy())
    
#     test_acc = accuracy_score(test_t, test_p)
#     print(f"CLEAN TEST ACC: {test_acc:.4f} | Time: {time.time()-start_time:.2f}s")
#     return test_acc


In [15]:
def run_experiment(name, model_fn):
    print(f"\n{'='*20} {name} {'='*20}")
    kf = KFold(n_splits=5, shuffle=True, random_state=1)
    
    fold_metrics = [] 
    start_time = time.time()

    for fold, (train_idx, val_idx) in enumerate(kf.split(train_val_dataset)):
        print(f"\n--- Starting Fold {fold+1} ---")
        t_loader = DataLoader(Subset(train_val_dataset, train_idx), batch_size=32, shuffle=True)
        v_loader = DataLoader(Subset(train_val_dataset, val_idx), batch_size=32, shuffle=False)
        
        # reset weights to prevent leakage
        model = model_fn().to(device)
        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
        criterion = nn.CrossEntropyLoss()

        # training Loop
        for epoch in range(3):
            model.train()
            for imgs, labels in t_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                optimizer.zero_grad()
                loss = criterion(model(imgs), labels)
                loss.backward()
                optimizer.step()
        
        # fold Validation
        model.eval()
        v_preds, v_probs, v_targets = [], [], []
        with torch.no_grad():
            for imgs, labels in v_loader:
                out = model(imgs.to(device))
                v_probs.extend(torch.softmax(out, dim=1).cpu().numpy())
                v_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                v_targets.extend(labels.numpy())
        
        # calculate Fold Metrics
        f_acc = accuracy_score(v_targets, v_preds)
        f_f1 = f1_score(v_targets, v_preds, average='weighted')
        f_auc = roc_auc_score(v_targets, v_probs, multi_class='ovr')
        
        fold_metrics.append({'acc': f_acc, 'f1': f_f1, 'auc': f_auc})
        print(f"Fold {fold+1} Results -> Acc: {f_acc:.4f} | F1: {f_f1:.4f} | AUC: {f_auc:.4f}")

    # final test set eval
    print(f"\nEvaluating final model on Kaggle Test Set...")
    model.eval()
    test_preds, test_probs, test_targets = [], [], []
    
    with torch.no_grad():
        for imgs, labels in test_loader:
            out = model(imgs.to(device))
            test_probs.extend(torch.softmax(out, dim=1).cpu().numpy())
            test_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
            test_targets.extend(labels.numpy())
    
    # final metrics
    t_acc = accuracy_score(test_targets, test_preds)
    t_f1 = f1_score(test_targets, test_preds, average='weighted')
    t_auc = roc_auc_score(test_targets, test_probs, multi_class='ovr')
    total_time = time.time() - start_time

    print(f"\n{'#'*10} SUMMARY: {name} {'#'*10}")
    print(f"Avg CV Acc: {np.mean([m['acc'] for m in fold_metrics]):.4f}")
    print(f"Final Test Acc: {t_acc:.4f}")
    print(f"Final Test F1:  {t_f1:.4f}")
    print(f"Final Test AUC: {t_auc:.4f}")
    print(f"Total Time:     {total_time:.2f}s")

    cm = confusion_matrix(test_targets, test_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_dirs, yticklabels=class_dirs)
    plt.title(f'Confusion Matrix: {name}')
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.show()
    
    return fold_metrics, t_acc

### Experiments

#### Curated

In [16]:
raw_curated = np.load('data/embs/finetuning_indexes_full_curated.npy').tolist()
curated_indices = [i for i in raw_curated if iNat36_label_df.iloc[i]['clean_label'] in global_label_to_idx]
train_val_dataset = InatPixelDataset(ds, curated_indices, iNat36_label_df, transform=preprocess)

In [ ]:
run_experiment("Curated - Partial Unfreeze", get_partial_unfreeze_model)


==================== Curated - Partial Unfreeze ====================

--- Starting Fold 1 ---


In [ ]:
run_experiment("Curated - LoRA", get_competitive_lora_model)

#### Full iNat

In [ ]:
train_val_dataset = InatPixelDataset(ds, all_valid_indices, iNat36_label_df, transform=preprocess)

In [ ]:
run_experiment("Full iNat - Partial Unfreeze", get_partial_unfreeze_model)

In [ ]:
run_experiment("Full iNat - LoRA", get_competitive_lora_model)

#### Random Sample

In [ ]:
target_size = 2347 # size matching original curated set
random_results = []
for trial in range(1, 11):
    np.random.seed(42 + trial)
    random_indices = np.random.choice(all_valid_indices, size=target_size, replace=False).tolist()
    train_val_dataset = InatPixelDataset(ds, random_indices, iNat36_label_df, transform=preprocess)
    random_results.append(run_experiment(f"Random Trial {trial}", get_partial_unfreeze_model))
    torch.cuda.empty_cache()

print(f"\nRandom Mean: {np.mean(random_results):.4f} (+/- {np.std(random_results):.4f})")